# Treinamento do detector de máscara de solda (YOLOv8) – versão Kaggle

**Projeto:** Verificador de EPIs de Solda com Visão Computacional (FSG + Marcopolo)
**Etapa:** 1.2 – Dataset + treinamento (RF31, RF32)

> **Foco da Fase 1 (decisão de 11/10/2026, com o orientador):** apenas **1 EPI, a máscara de solda**
> (também chamada de capacete de solda). Os demais EPIs ficam como trabalhos futuros: basta acrescentar as
> classes em `CLASSES_MODELO` (célula 1) para treiná-los.

O que este notebook faz, em ordem:

1. Baixa as **duas fontes de dados** (os seus forks no Roboflow): *Welding-PPE* e *Welding Helmet*.
2. Confere se nenhuma versão é de "objetos isolados" (recortes).
3. **Remapeia** as classes de cada fonte para `mascara`. As caixas de outros EPIs são removidas, e as imagens
   sem máscara viram **exemplos negativos**, que ensinam o modelo a não confundir uma luva ou um avental com uma máscara.
4. Acrescenta fotos de **pessoas comuns sem máscara** (COCO e, opcionalmente, fotos próprias) como negativos.
   Junta as fontes, faz a **divisão estratificada** treino/validação/teste (70/20/10) e a augmentation de treino.
5. Gera o **relatório de balanceamento**.
6. Treina **YOLOv8n** e **YOLOv8s** (exigidos pelo projeto) e o **YOLO26n** (comparação), com `imgsz` **640** e **960**.
7. Avalia no conjunto de **teste**: precisão, recall, mAP@0.5 e mAP@0.5:0.95 + matriz de confusão.
8. Exporta o melhor modelo como `epi_solda_best.pt`.

**Antes de rodar (Kaggle):** no painel da direita, *Session options* → **Accelerator: GPU T4 x2** e **Internet: On**;
*Add-ons → Secrets* → `ROBOFLOW_API_KEY` anexada a este notebook. Para continuar um treino começado no Colab,
anexe a pasta `treinos` como *Input* (seção 2b).

**Para rodar com o navegador fechado:** *Save Version* → **Save & Run All (Commit)**. O resultado fica na aba *Output* da versão.

### Atribuição dos datasets (CC BY 4.0)

**Welding-PPE**, por *Welding PPE*, Roboflow Universe, CC BY 4.0 – https://universe.roboflow.com/welding-ppe/welding-ppe

```bibtex
@misc{ welding-ppe_dataset,
  title = { Welding-PPE Dataset },
  type = { Open Source Dataset },
  author = { Welding PPE },
  howpublished = { \url{ https://universe.roboflow.com/welding-ppe/welding-ppe } },
  url = { https://universe.roboflow.com/welding-ppe/welding-ppe },
  journal = { Roboflow Universe },
  publisher = { Roboflow },
  year = { 2023 },
  month = { jun },
  note = { visited on 2026-10-10 },
}
```

**Welding Helmet**, por *UTP*, Roboflow Universe, CC BY 4.0 – https://universe.roboflow.com/utp-jtbn5/welding-helmet

```bibtex
@misc{ welding-helmet_dataset,
  title = { Welding Helmet Dataset },
  type = { Open Source Dataset },
  author = { UTP },
  howpublished = { \url{ https://universe.roboflow.com/utp-jtbn5/welding-helmet } },
  url = { https://universe.roboflow.com/utp-jtbn5/welding-helmet },
  journal = { Roboflow Universe },
  publisher = { Roboflow },
  year = { 2025 },
  month = { apr },
  note = { visited on 2026-10-10 },
}
```

## 0. Decisões sobre os dados (resumo)

- **Forks em vez das versões públicas.** As versões públicas do Welding-PPE usam *Isolate Objects* (cada caixa vira
  um recorte), e a do Welding Helmet tem rotações de 90°. Por isso usamos *forks* com versões geradas só com *Auto-Orient*.
- **Divisão feita aqui, não no Roboflow.** O *Rebalance* do Roboflow deixou só 2 máscaras no teste da primeira rodada (v1).
  Aqui a divisão é **estratificada** (seção 6b), com semente fixa.
- **Cópias aumentadas não vazam.** Se a versão do Roboflow tiver cópias aumentadas, mantemos só 1 de cada imagem
  original, para não ter a mesma foto no treino e no teste. Blur e exposição são aplicados aqui, só no treino.
- **Negativos limitados.** O Welding-PPE tem centenas de fotos sem máscara (luvas, botas, aventais). Usamos parte delas
  como negativos, limitadas a `FRACAO_MAX_NEGATIVOS` do total, para que não dominem o treino.
- **Pessoas sem máscara (rodada v3).** Na webcam, o modelo da v2 confundia rosto inclinado/de perfil com máscara.
  Por isso entram fotos de pessoas comuns do COCO como negativos (seção 6a).

Detalhes e passo a passo: `docs/dataset_welding_ppe.md`.

## 1. Configuração

Preencha `workspace`, `projeto` e `versao` de cada fonte com os dados dos **seus forks**. Eles aparecem no código de
download do Roboflow: `rf.workspace("...").project("...").version(N)`.

A chave da API **não** vai no código: cadastre-a em *Add-ons → Secrets* (menu do Kaggle) com o nome
`ROBOFLOW_API_KEY` e marque a caixa ao lado para anexá-la a este notebook.

In [ ]:
# ===== Fontes de dados (forks no SEU workspace do Roboflow) =====
FONTES = [
    {"nome": "Welding-PPE",    "workspace": "leonardo-matte", "projeto": "welding-ppe-9n9br", "versao": 1},
    {"nome": "Welding Helmet", "workspace": "leonardo-matte", "projeto": "welding-helmet-nqk1c", "versao": 1},
]

# ===== O que o modelo aprende =====
CLASSES_MODELO = ["mascara"]    # foco da Fase 1; no futuro: + "luvas", "avental", "macacao", "calcado"
RODADA = "v3"                   # nome desta rodada de treino (pasta de saída e em docs/resultados_treino/)

# ===== Dados =====
PROPORCOES = {"train": 0.70, "val": 0.20, "test": 0.10}
AUMENTOS_POR_IMAGEM = 1         # cópias extras de cada imagem de treino (blur + exposição); 1 = dobra o treino
FRACAO_MAX_NEGATIVOS = 0.30     # limite para as imagens sem máscara que vêm junto das fontes (luvas, botas...)

# ===== Exemplos negativos extras: pessoas SEM máscara de solda (seção 6a) =====
# Corrigem o falso positivo visto na webcam (rosto inclinado ou de perfil detectado como máscara, rodada v2).
USAR_COCO_PESSOAS = True        # baixa fotos de pessoas do COCO 2017 (validação) automaticamente
MAX_COCO_PESSOAS  = 1500        # quantas fotos usar
AREA_MIN_PESSOA   = 0.05        # só fotos com uma pessoa ocupando >= 5% da imagem (cabeça grande o bastante)
PASTAS_NEGATIVOS  = []          # fotos próprias sem máscara anexadas como Input, ex.: ["/kaggle/input/epi-solda-negativos-webcam"]

# ===== Treino =====
EXPERIMENTOS = [                # (modelo base, imgsz). A comparação completa (YOLOv8 n/s, 640/960, YOLO26n) foi
    ("yolov8n.pt", 640),        # feita na v2 (RF32). Na 1ª execução da v3, com os negativos do COCO, o yolov8n_640 teve
]                               # 7,9% de falso positivo contra 27,9% do yolo26n_640 -> treinamos só ele
EPOCAS    = 100                 # máximo de épocas
PACIENCIA = 25                  # para antes se o mAP de validação não melhorar por N épocas
SEMENTE   = 42                  # resultados reproduzíveis

# Versões fixadas (as mesmas do requirements.txt do projeto)
VERSAO_ULTRALYTICS = "8.4.175"
VERSAO_ROBOFLOW    = "1.7.1"

# Kaggle: tudo que fica em /kaggle/working vira a "saída" (aba Output) da execução.
# O nome PASTA_DRIVE foi mantido igual ao do notebook do Colab para o restante do código ser o mesmo.
PASTA_DRIVE = "/kaggle/working/epi_solda"
PASTA_TMP   = "/tmp/epi_solda"     # dados temporários (downloads e dataset), fora da saída

## 2. Ambiente: GPU e bibliotecas

No Kaggle não há Google Drive: os treinos ficam em `/kaggle/working/epi_solda/<rodada>/treinos` e são salvos na
aba *Output* da execução. Os experimentos concluídos são pulados e o que estava pela metade é **retomado** do
último checkpoint, desde que a pasta `treinos` anterior esteja anexada como *Input* (seção 2b).

In [ ]:
import subprocess, sys

# Instala as versões fixadas (silencioso)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"ultralytics=={VERSAO_ULTRALYTICS}", f"roboflow=={VERSAO_ROBOFLOW}"], check=True)

import torch
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ATENÇÃO: sem GPU! Ative em Session options → Accelerator → GPU T4 x2. Em CPU o treino leva horas.")

from pathlib import Path
PASTA_DRIVE = Path(PASTA_DRIVE) / RODADA
PASTA_TREINOS = PASTA_DRIVE / "treinos"
PASTA_TREINOS.mkdir(parents=True, exist_ok=True)
print("Treinos serão salvos em:", PASTA_TREINOS)

## 2b. Continuar de onde parou (progresso do Colab/Drive ou de uma execução anterior do Kaggle)

Se você já treinou parte dos experimentos em outro lugar, envie a pasta `treinos` daquela rodada como um
**Dataset** do Kaggle e anexe-a em *Input → Add Input*. Esta célula procura, dentro de `/kaggle/input`, pastas com
o nome dos experimentos (`yolov8n_640`, `yolov8s_640`, ...) que tenham a subpasta `weights`, e as copia para a pasta
de treinos desta sessão:

- experimentos com `CONCLUIDO.txt` são **pulados** no treino;
- o experimento com `weights/last.pt` e sem `CONCLUIDO.txt` é **retomado** (os caminhos do Colab guardados no
  checkpoint são corrigidos automaticamente).

Também funciona com a saída de uma execução anterior deste notebook: *Add Input → Notebook Output*.

**Arquivos convertidos pelo Google Drive:** se o Drive estiver configurado para converter uploads, `results.csv`
vira `results.xlsx` e `CONCLUIDO.txt` vira `CONCLUIDO.docx`. Esta célula desfaz essa conversão.

In [ ]:
import shutil
import pandas as pd

esperados = {f"{Path(m).stem}_{s}" for m, s in EXPERIMENTOS}
importados = []
entradas = Path("/kaggle/input")
for pasta in (sorted(p for p in entradas.rglob("*") if p.is_dir()) if entradas.exists() else []):
    if pasta.name in esperados and (pasta / "weights").is_dir() and not (PASTA_TREINOS / pasta.name).exists():
        destino = PASTA_TREINOS / pasta.name
        shutil.copytree(pasta, destino)
        # Desfaz a conversão do Google Drive (.csv -> .xlsx e .txt -> .docx)
        if (destino / "results.xlsx").exists() and not (destino / "results.csv").exists():
            pd.read_excel(destino / "results.xlsx").to_csv(destino / "results.csv", index=False)
        if (destino / "CONCLUIDO.docx").exists() and not (destino / "CONCLUIDO.txt").exists():
            (destino / "CONCLUIDO.txt").write_text("ok", encoding="utf-8")
        pesos = {p.name: p.stat().st_size / 1e6 for p in sorted((destino / "weights").glob("*.pt"))}
        estado = "concluído" if (destino / "CONCLUIDO.txt").exists() else "a retomar"
        alerta = "  <- ATENÇÃO: peso com menos de 1 MB (arquivo incompleto?)" if min(pesos.values(), default=0) < 1 else ""
        importados.append(f"{pasta.name} ({estado}) | " + ", ".join(f"{n}: {t:.1f} MB" for n, t in pesos.items()) + alerta)

print("Experimentos importados:" if importados else "Nenhum progresso anterior encontrado em /kaggle/input.")
for linha in importados:
    print("  ", linha)

## 3. Download das fontes via API do Roboflow (RF31)

Baixamos no formato **YOLOv8**: uma pasta por split (`train/`, `valid/`, `test/`), cada uma com
`images/` e `labels/`. Cada arquivo `.txt` de label tem uma linha por objeto:

```
<id_da_classe> <x_centro> <y_centro> <largura> <altura>     (valores de 0 a 1, relativos à imagem)
```

In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

pendentes = [f["nome"] for f in FONTES if "PREENCHER" in (f["workspace"], f["projeto"])]
if pendentes:
    raise ValueError(f"Preencha workspace/projeto na célula 1 para: {pendentes}")

chave = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")   # lida do cofre de Secrets, nunca escrita no código
rf = Roboflow(api_key=chave)

PASTAS_FONTES: list[Path] = []
for i, fonte in enumerate(FONTES):
    versao = rf.workspace(fonte["workspace"]).project(fonte["projeto"]).version(fonte["versao"])
    baixado = versao.download("yolov8", location=f"{PASTA_TMP}/fontes/f{i}", overwrite=True)
    PASTAS_FONTES.append(Path(baixado.location))
    print(f"[f{i}] {fonte['nome']} baixado em {baixado.location}")

## 4. Funções auxiliares de leitura do dataset

In [ ]:
import yaml

SPLITS = {"train": "train", "val": "valid", "test": "test"}   # nome Ultralytics -> pasta Roboflow


def ler_nomes_classes(pasta: Path) -> list[str]:
    # Lê a lista de nomes de classes do data.yaml (pode ser lista ou dicionário {id: nome}).
    dados = yaml.safe_load((pasta / "data.yaml").read_text(encoding="utf-8"))
    nomes = dados["names"]
    if isinstance(nomes, dict):
        nomes = [nomes[i] for i in sorted(nomes)]
    return list(nomes)


def splits_existentes(pasta: Path) -> dict[str, Path]:
    # Retorna {split: pasta} só para os splits que existem no download.
    return {s: pasta / p for s, p in SPLITS.items() if (pasta / p / "labels").is_dir()}


def ler_label(arquivo: Path) -> list[list[float]]:
    # Lê um .txt YOLO e devolve uma lista de linhas numéricas [classe, ...coordenadas].
    linhas = []
    for linha in arquivo.read_text(encoding="utf-8").splitlines():
        partes = linha.split()
        if partes:
            linhas.append([int(partes[0])] + [float(v) for v in partes[1:]])
    return linhas


def caixa_de(linha: list[float]) -> tuple[float, float, float, float]:
    # Devolve (xc, yc, w, h). Se a linha for polígono (segmentação), converte para a caixa envolvente.
    valores = linha[1:]
    if len(valores) == 4:
        return tuple(valores)
    xs, ys = valores[0::2], valores[1::2]
    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
    return ((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)


for i, pasta in enumerate(PASTAS_FONTES):
    print(f"[f{i}] classes originais: {ler_nomes_classes(pasta)} | splits: {list(splits_existentes(pasta))}")

## 5. Verificação: alguma versão é de "objetos isolados"?

Se a maioria das imagens tiver **uma única caixa ocupando quase a imagem toda**, a versão pode ter sido gerada com
"Isolate Objects". É só um **aviso**, porque as duas fontes têm muitas fotos de produto (um objeto sobre fundo neutro).
A confirmação definitiva é no Roboflow: em *Preprocessing* deve aparecer só "Auto-Orient".

In [ ]:
def verificar_objetos_isolados(pasta: Path) -> float:
    # Retorna a fração de imagens (todos os splits) com 1 só caixa cobrindo mais de 60% da imagem.
    labels = [a for p in splits_existentes(pasta).values() for a in (p / "labels").glob("*.txt")]
    suspeitas = 0
    for arq in labels:
        linhas = ler_label(arq)
        if len(linhas) == 1:
            _, _, w, h = caixa_de(linhas[0])
            if w * h > 0.60:
                suspeitas += 1
    return suspeitas / max(len(labels), 1)


for i, pasta in enumerate(PASTAS_FONTES):
    fracao = verificar_objetos_isolados(pasta)
    aviso = "  <- ATENÇÃO: confira o Preprocessing no Roboflow" if fracao > 0.5 else ""
    print(f"[f{i}] {FONTES[i]['nome']}: {fracao:.0%} das imagens com 1 caixa ocupando >60%{aviso}")

## 6. Remapeamento das classes (RF03, RF31)

Cada fonte tem os seus nomes de classe. Convertemos todos para os nomes internos do projeto e mantemos só as
classes de `CLASSES_MODELO`.

| Classe original | Classe interna | Fonte |
|---|---|---|
| Welding Mask | `mascara` | Welding-PPE |
| helmet / Helmet | `mascara` | Welding-PPE e Welding Helmet. **Verificado visualmente**: são máscaras (capacetes) de solda, não capacetes de segurança |
| Welding Gloves, welding-glove | `luvas` | Welding-PPE (fora do foco: removida) |
| Welding Apron, apron | `avental` | Welding-PPE (fora do foco: removida) |
| Welding Coverall | `macacao` | Welding-PPE (fora do foco: removida) |
| Safety Shoes, welding-shose | `calcado` | Welding-PPE (fora do foco: removida) |

Uma imagem que só tinha caixas removidas (por exemplo, a foto de uma luva) fica **sem objetos**: é um exemplo negativo.

In [ ]:
import shutil

# Nome original (minúsculo, sem espaços nas pontas) -> classe interna do projeto
MAPEAMENTO = {
    "welding mask": "mascara",
    "helmet": "mascara",
    "welding gloves": "luvas",
    "welding-glove": "luvas",
    "welding apron": "avental",
    "apron": "avental",
    "welding coverall": "macacao",
    "safety shoes": "calcado",
    "welding-shose": "calcado",
}
MAPEAMENTO.update({c: c for c in ["mascara", "luvas", "avental", "macacao", "calcado", "capacete"]})


def remapear_fonte(origem: Path, destino: Path) -> dict:
    # Copia as imagens e reescreve os labels só com as classes de CLASSES_MODELO. Retorna um resumo.
    nomes_originais = ler_nomes_classes(origem)
    id_novo = {}
    for i, nome in enumerate(nomes_originais):
        interno = MAPEAMENTO.get(nome.strip().lower())
        id_novo[i] = CLASSES_MODELO.index(interno) if interno in CLASSES_MODELO else None

    resumo = {"imagens": 0, "imagens_com_objeto": 0, "caixas": 0, "removidas_fora_do_foco": {}, "desconhecidas": {}}
    if destino.exists():
        shutil.rmtree(destino)
    for split, pasta_split in splits_existentes(origem).items():
        (destino / SPLITS[split] / "labels").mkdir(parents=True)
        shutil.copytree(pasta_split / "images", destino / SPLITS[split] / "images")
        for arq in (pasta_split / "labels").glob("*.txt"):
            novas = []
            for linha in ler_label(arq):
                novo = id_novo.get(linha[0])
                if novo is None:
                    nome = nomes_originais[linha[0]]
                    grupo = "removidas_fora_do_foco" if nome.strip().lower() in MAPEAMENTO else "desconhecidas"
                    resumo[grupo][nome] = resumo[grupo].get(nome, 0) + 1
                    continue
                xc, yc, w, h = caixa_de(linha)
                novas.append(f"{novo} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
            (destino / SPLITS[split] / "labels" / arq.name).write_text("\n".join(novas), encoding="utf-8")
            resumo["imagens"] += 1
            resumo["imagens_com_objeto"] += bool(novas)
            resumo["caixas"] += len(novas)
    return resumo


NOMES_FONTES = {f"f{i}": f["nome"] for i, f in enumerate(FONTES)}   # prefixo -> nome (para os relatórios)
PREFIXOS_NEGATIVOS: set[str] = set()                                  # fontes só de negativos (seção 6a)
PASTAS_REMAP: list[Path] = []
for i, pasta in enumerate(PASTAS_FONTES):
    destino = Path(f"{PASTA_TMP}/remap/f{i}")
    resumo = remapear_fonte(pasta, destino)
    PASTAS_REMAP.append(destino)
    print(f"[f{i}] {FONTES[i]['nome']}: {resumo['imagens']} imagens, {resumo['imagens_com_objeto']} com "
          f"{CLASSES_MODELO}, {resumo['caixas']} caixas mantidas")
    print(f"      removidas (fora do foco): {resumo['removidas_fora_do_foco'] or 'nenhuma'}")
    if resumo["desconhecidas"]:
        print(f"      ATENÇÃO, classes sem mapeamento (removidas): {resumo['desconhecidas']}")

## 6a. Exemplos negativos extras: pessoas sem máscara de solda

**Por que:** na rodada v2, o teste na webcam mostrou o **rosto descoberto** sendo detectado como máscara quando a
cabeça estava inclinada ou de perfil (confiança 0,67 e 0,88). Os únicos negativos do treino eram fotos de luvas,
botas e aventais: o modelo nunca tinha visto uma cabeça de pessoa **sem** máscara.

**O que fazemos:** acrescentamos fotos de **pessoas comuns** (de frente, de perfil, inclinadas, de costas, de boné,
de capacete de moto) **sem nenhuma caixa**. Uma imagem sem caixa ensina ao modelo: "nada aqui é máscara de solda".
Esses negativos **não precisam de anotação** e não entram no limite `FRACAO_MAX_NEGATIVOS`.

- **COCO 2017** (conjunto de validação, imagens com pessoas): baixado automaticamente das fontes oficiais.
  Usamos só fotos em que uma pessoa ocupa pelo menos `AREA_MIN_PESSOA` da imagem, para a cabeça aparecer bem.
- **Fotos próprias** (opcional): pastas em `PASTAS_NEGATIVOS`, por exemplo frames da webcam extraídos com
  `ferramentas/extrair_frames.py`. Só use fotos de pessoas que autorizaram.

*COCO: Lin et al., "Microsoft COCO: Common Objects in Context", ECCV 2014 – https://cocodataset.org.
Anotações sob CC BY 4.0; as imagens seguem as licenças do Flickr indicadas no próprio COCO.*

In [ ]:
import random
import urllib.request
import zipfile

URL_COCO_ROTULOS = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017labels.zip"
URL_COCO_IMAGENS = "http://images.cocodataset.org/zips/val2017.zip"
EXTENSOES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
BASE_NEGATIVOS = Path(f"{PASTA_TMP}/negativos")


def baixar(url: str, destino: Path) -> Path:
    # Baixa um arquivo uma única vez (se já existir, reaproveita).
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        print("Baixando", url, "...")
        urllib.request.urlretrieve(url, destino)
    return destino


def preparar_pasta_negativa(destino: Path) -> tuple[Path, Path]:
    # Cria a estrutura de uma "fonte" só com negativos (tudo em train/; a divisão é refeita na seção 6b).
    shutil.rmtree(destino, ignore_errors=True)
    pasta_img, pasta_lbl = destino / "train" / "images", destino / "train" / "labels"
    pasta_img.mkdir(parents=True); pasta_lbl.mkdir(parents=True)
    return pasta_img, pasta_lbl


def negativos_coco(destino: Path) -> int:
    # Seleciona fotos do COCO val2017 com pessoas grandes e grava sem nenhuma caixa (label vazio).
    rotulos = zipfile.ZipFile(baixar(URL_COCO_ROTULOS, BASE_NEGATIVOS / "coco2017labels.zip"))
    candidatas = []
    for nome in rotulos.namelist():
        if "/labels/val2017/" in nome and nome.endswith(".txt"):
            pessoas = [l.split() for l in rotulos.read(nome).decode().splitlines() if l.startswith("0 ")]
            if pessoas and max(float(p[3]) * float(p[4]) for p in pessoas) >= AREA_MIN_PESSOA:
                candidatas.append(Path(nome).stem)
    candidatas.sort()
    random.Random(SEMENTE).shuffle(candidatas)
    escolhidas = set(candidatas[:MAX_COCO_PESSOAS])

    imagens = zipfile.ZipFile(baixar(URL_COCO_IMAGENS, BASE_NEGATIVOS / "val2017.zip"))
    pasta_img, pasta_lbl = preparar_pasta_negativa(destino)
    gravadas = 0
    for nome in imagens.namelist():
        stem = Path(nome).stem
        if nome.endswith(".jpg") and stem in escolhidas:
            (pasta_img / f"coco_{stem}.jpg").write_bytes(imagens.read(nome))
            (pasta_lbl / f"coco_{stem}.txt").write_text("", encoding="utf-8")
            gravadas += 1
    print(f"COCO: {len(candidatas)} fotos com pessoa grande; usando {gravadas}")
    return gravadas


def negativos_pasta(origem: Path, destino: Path) -> int:
    # Copia todas as imagens de uma pasta (e subpastas) como negativos.
    arquivos = [p for p in sorted(Path(origem).rglob("*")) if p.suffix.lower() in EXTENSOES]
    pasta_img, pasta_lbl = preparar_pasta_negativa(destino)
    for k, arq in enumerate(arquivos):
        nome = f"{arq.stem}_{k}"
        shutil.copy(arq, pasta_img / f"{nome}{arq.suffix.lower()}")
        (pasta_lbl / f"{nome}.txt").write_text("", encoding="utf-8")
    return len(arquivos)


extras = ([("COCO pessoas", None)] if USAR_COCO_PESSOAS else []) + [(f"pasta {p}", p) for p in PASTAS_NEGATIVOS]
for nome, origem in extras:
    prefixo = f"f{len(PASTAS_REMAP)}"
    destino = Path(f"{PASTA_TMP}/remap/{prefixo}")
    quantidade = negativos_coco(destino) if origem is None else negativos_pasta(Path(origem), destino)
    if quantidade == 0:
        print(f"ATENÇÃO: nenhuma imagem encontrada para {nome}")
        continue
    PASTAS_REMAP.append(destino)
    NOMES_FONTES[prefixo] = nome
    PREFIXOS_NEGATIVOS.add(prefixo)
    print(f"[{prefixo}] {nome}: {quantidade} imagens negativas")

## 6b. Junção das fontes, divisão estratificada e augmentation de treino

**Divisão estratificada:** juntamos as imagens das duas fontes e as distribuímos em 70% / 20% / 10%, de modo que
**cada classe** (e os negativos) também fique perto de 70/20/10. As imagens com as classes mais raras são distribuídas
primeiro (estratificação iterativa), sempre com a mesma semente.

**Sem vazamento:** as cópias de uma mesma foto original (geradas pelo Roboflow ou clonadas entre projetos) são
reconhecidas pelo nome e mantidas **uma vez só**.

**Augmentation de treino:** `AUMENTOS_POR_IMAGEM` cópias de cada imagem de **treino** com exposição aleatória (±25%) e
desfoque (até 2,5 px). Validação e teste ficam intactos. As caixas não mudam, porque essas transformações não movem os objetos.

In [ ]:
import random
import cv2
import numpy as np
from collections import defaultdict


def grupo_original(nome: str) -> str:
    # O Roboflow nomeia as cópias como "<original>_jpg.rf.<hash>.jpg"; o grupo é a parte antes de ".rf.".
    return nome.split(".rf.")[0]


def dividir_estratificado(origens: list[Path]) -> dict[str, list[tuple[Path, Path, str]]]:
    # Junta as imagens de todas as fontes e redistribui por estratificação iterativa.
    # Cada item é (imagem, label, prefixo da fonte).
    grupos: dict[str, list[tuple[Path, Path, str]]] = defaultdict(list)
    for i, origem in enumerate(origens):
        for pasta_split in splits_existentes(origem).values():
            for img in sorted((pasta_split / "images").iterdir()):
                grupos[grupo_original(img.name)].append((img, pasta_split / "labels" / (img.stem + ".txt"), f"f{i}"))

    repetidos = sum(len(v) - 1 for v in grupos.values())
    print(f"Cópias da mesma foto original descartadas: {repetidos}")
    itens = [lista[0] for lista in grupos.values()]
    classes_do_item = [{int(l[0]) for l in ler_label(lbl)} if lbl.exists() else set() for _, lbl, _ in itens]

    # Negativos que vêm junto das fontes (luvas, botas...) são limitados; os negativos extras (6a) entram todos
    rnd = random.Random(SEMENTE)
    positivos = [k for k, c in enumerate(classes_do_item) if c]
    extras = [k for k, c in enumerate(classes_do_item) if not c and itens[k][2] in PREFIXOS_NEGATIVOS]
    incidentais = [k for k, c in enumerate(classes_do_item) if not c and itens[k][2] not in PREFIXOS_NEGATIVOS]
    max_neg = int(FRACAO_MAX_NEGATIVOS * len(positivos) / (1 - FRACAO_MAX_NEGATIVOS))
    rnd.shuffle(incidentais)
    print(f"Imagens com objeto: {len(positivos)} | negativos das fontes: {len(incidentais)} disponíveis, "
          f"{min(len(incidentais), max_neg)} usados | negativos extras (seção 6a): {len(extras)}")
    escolhidos = positivos + incidentais[:max_neg] + extras

    total_por_classe = defaultdict(int)
    for k in escolhidos:
        for c in classes_do_item[k]:
            total_por_classe[c] += 1
    desejado = {s: {c: n * p for c, n in total_por_classe.items()} for s, p in PROPORCOES.items()}
    desejado_imgs = {s: len(escolhidos) * p for s, p in PROPORCOES.items()}
    atual = {s: defaultdict(int) for s in PROPORCOES}
    atual_imgs = {s: 0 for s in PROPORCOES}

    rnd.shuffle(escolhidos)
    # Primeiro as imagens cuja classe mais rara é a mais rara de todas; negativos por último.
    escolhidos.sort(key=lambda k: min((total_por_classe[c] for c in classes_do_item[k]), default=10**9))

    divisao: dict[str, list[tuple[Path, Path, str]]] = {s: [] for s in PROPORCOES}
    for k in escolhidos:
        classes = classes_do_item[k]
        if classes:
            rara = min(classes, key=lambda c: total_por_classe[c])
            chave = lambda s: (desejado[s][rara] - atual[s][rara], desejado_imgs[s] - atual_imgs[s])
        else:
            chave = lambda s: desejado_imgs[s] - atual_imgs[s]
        split = max(PROPORCOES, key=chave)
        divisao[split].append(itens[k])
        atual_imgs[split] += 1
        for c in classes:
            atual[split][c] += 1
    return divisao


def aumentar(img: np.ndarray, rnd: random.Random) -> np.ndarray:
    # Exposição aleatória (±25%) + desfoque gaussiano de até 2,5 px.
    saida = np.clip(img.astype(np.float32) * rnd.uniform(0.75, 1.25), 0, 255).astype(np.uint8)
    raio = rnd.uniform(0, 2.5)
    if raio >= 0.5:
        k = 2 * int(round(raio)) + 1
        saida = cv2.GaussianBlur(saida, (k, k), 0)
    return saida


def montar_dataset_final(divisao: dict[str, list[tuple[Path, Path, str]]], destino: Path) -> None:
    # Copia as imagens para destino/<split>/ (nome prefixado com a fonte), gera as cópias aumentadas do treino
    # e escreve o data.yaml.
    rnd = random.Random(SEMENTE)
    if destino.exists():
        shutil.rmtree(destino)
    for split, itens in divisao.items():
        pasta_img, pasta_lbl = destino / SPLITS[split] / "images", destino / SPLITS[split] / "labels"
        pasta_img.mkdir(parents=True); pasta_lbl.mkdir(parents=True)
        for img, lbl, prefixo in itens:
            nome = f"{prefixo}_{img.stem}"
            shutil.copy(img, pasta_img / f"{nome}{img.suffix}")
            texto = lbl.read_text(encoding="utf-8") if lbl.exists() else ""
            (pasta_lbl / f"{nome}.txt").write_text(texto, encoding="utf-8")
            if split != "train" or AUMENTOS_POR_IMAGEM == 0:
                continue
            original = cv2.imread(str(img))
            if original is None:
                continue
            for k in range(AUMENTOS_POR_IMAGEM):
                cv2.imwrite(str(pasta_img / f"{nome}_aug{k}.jpg"), aumentar(original, rnd))
                (pasta_lbl / f"{nome}_aug{k}.txt").write_text(texto, encoding="utf-8")
    data_yaml = {"path": str(destino), "names": {i: n for i, n in enumerate(CLASSES_MODELO)},
                 "train": "train/images", "val": "valid/images", "test": "test/images"}
    (destino / "data.yaml").write_text(yaml.safe_dump(data_yaml, allow_unicode=True, sort_keys=False),
                                       encoding="utf-8")


divisao = dividir_estratificado(PASTAS_REMAP)
PASTA_DATASET = Path(f"{PASTA_TMP}/dataset")
montar_dataset_final(divisao, PASTA_DATASET)
DATA_YAML = PASTA_DATASET / "data.yaml"

print("Imagens originais por split:", {s: len(v) for s, v in divisao.items()})
print("Treino com augmentation:", len(list((PASTA_DATASET / "train" / "images").iterdir())), "imagens")

## 7. Relatório de balanceamento (RF31)

Contamos só as imagens **originais** (sem as cópias aumentadas do treino), separando por fonte (os nomes de cada
prefixo `f0`, `f1`, ... aparecem na saída das seções 6 e 6a). A **área média** mostra o tamanho típico da máscara na imagem: objetos pequenos são mais
difíceis e se beneficiam de `imgsz` maior.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def relatorio_balanceamento(pasta: Path) -> pd.DataFrame:
    # Conta instâncias por classe e split, imagens negativas e tamanho médio das caixas.
    registros = []
    for split, pasta_split in splits_existentes(pasta).items():
        # Só imagens originais (as cópias "_aug" do treino distorceriam a contagem)
        arquivos = [a for a in (pasta_split / "labels").glob("*.txt") if "_aug" not in a.stem]
        for arq in arquivos:
            fonte = arq.stem.split("_")[0]
            linhas = ler_label(arq)
            if not linhas:
                registros.append({"split": split, "classe": "(negativo: sem objetos)", "fonte": fonte,
                                  "imagem": arq.stem, "area": None})
            for linha in linhas:
                _, _, w, h = caixa_de(linha)
                registros.append({"split": split, "classe": CLASSES_MODELO[linha[0]], "fonte": fonte,
                                  "imagem": arq.stem, "area": w * h})
    df = pd.DataFrame(registros)
    instancias = df.pivot_table(index="classe", columns="split", values="imagem", aggfunc="count", fill_value=0)
    por_fonte = df.pivot_table(index="classe", columns="fonte", values="imagem", aggfunc="count", fill_value=0)
    imagens = df.groupby("classe")["imagem"].nunique().rename("imagens")
    area = (df.groupby("classe")["area"].mean() * 100).round(2).rename("area_media_%_da_imagem")
    tabela = instancias.join(por_fonte).join(imagens).join(area)
    tabela["total"] = instancias.sum(axis=1)
    return tabela.sort_values("total", ascending=False)


balanceamento = relatorio_balanceamento(PASTA_DATASET)
display(balanceamento)
balanceamento.to_csv(PASTA_DRIVE / "balanceamento_classes.csv")

ax = balanceamento[["train", "val", "test"]].plot.bar(title="Instâncias / imagens por split", rot=0)
ax.set_ylabel("quantidade")
plt.tight_layout(); plt.savefig(PASTA_DRIVE / "balanceamento_classes.png"); plt.show()

### Amostras após o remapeamento
Confira visualmente: as caixas `mascara` devem estar sobre máscaras de solda, e os negativos **não** devem ter
nenhuma máscara visível. Se um negativo tiver máscara, ela não foi anotada na fonte original. Anote o nome
do arquivo para registrar como limitação.

In [ ]:
CORES = [(220, 53, 69), (13, 110, 253), (25, 135, 84), (255, 193, 7), (111, 66, 193)]


def mostrar_amostras(pasta: Path, por_linha: int = 4) -> None:
    # Mostra imagens de treino de cada classe do modelo e alguns negativos, com as caixas desenhadas.
    random.seed(SEMENTE)
    labels = [a for a in (pasta / "train" / "labels").glob("*.txt") if "_aug" not in a.stem]
    com_objeto = [a for a in labels if ler_label(a)]
    sem_objeto = [a for a in labels if not ler_label(a)]
    linhas_grade = [(c, [a for a in com_objeto if any(l[0] == i for l in ler_label(a))])
                    for i, c in enumerate(CLASSES_MODELO)]
    for prefixo, nome_fonte in NOMES_FONTES.items():
        if prefixo in PREFIXOS_NEGATIVOS:
            linhas_grade.append((f"negativos ({nome_fonte})", [a for a in sem_objeto if a.stem.startswith(prefixo + "_")]))
        else:
            linhas_grade.append((f"{CLASSES_MODELO[0]} ({nome_fonte})",
                                 [a for a in com_objeto if a.stem.startswith(prefixo + "_")]))
    linhas_grade.append(("negativos das fontes (luvas, botas...)",
                         [a for a in sem_objeto if a.stem.split("_")[0] not in PREFIXOS_NEGATIVOS]))

    fig, eixos = plt.subplots(len(linhas_grade), por_linha, figsize=(4 * por_linha, 3.2 * len(linhas_grade)),
                              squeeze=False)
    for r, (titulo, candidatos) in enumerate(linhas_grade):
        for j in range(por_linha):
            ax = eixos[r][j]; ax.axis("off")
            if j >= len(candidatos):
                continue
            arq = random.choice(candidatos)
            img_path = next((pasta / "train" / "images").glob(arq.stem + ".*"))
            img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
            alt, larg = img.shape[:2]
            for linha in ler_label(arq):
                xc, yc, w, h = caixa_de(linha)
                p1 = (int((xc - w / 2) * larg), int((yc - h / 2) * alt))
                p2 = (int((xc + w / 2) * larg), int((yc + h / 2) * alt))
                cv2.rectangle(img, p1, p2, CORES[linha[0]], 3)
                cv2.putText(img, CLASSES_MODELO[linha[0]], (p1[0], max(p1[1] - 5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, CORES[linha[0]], 2)
            ax.imshow(img); ax.set_title(f"{titulo}: {arq.stem[:28]}", fontsize=8)
    plt.tight_layout(); plt.show()


mostrar_amostras(PASTA_DATASET)

## 8. Treinamento (RF32)

### Augmentations (o que cada uma faz e por que ajuda na solda)

*Augmentation* é gerar, durante o treino, variações artificiais das imagens. O modelo vê "imagens novas"
a cada época e generaliza melhor.

| Augmentation | Onde | Parâmetro | Por que ajuda no posto de solda |
|---|---|---|---|
| Brilho/exposição | Notebook (6b) + Ultralytics | exposição ±25% (6b); `hsv_v=0.5` | O arco e a iluminação da fábrica mudam muito o brilho |
| Contraste/saturação | Ultralytics | `hsv_s=0.6` | Fumos "lavam" as cores; as máscaras têm cores e pinturas variadas |
| Desfoque (blur) | Notebook (6b) | blur até 2,5 px | Fumos, respingos no vidro da câmera, movimento da cabeça |
| Mosaic | Ultralytics | `mosaic=1.0` | Junta 4 imagens em uma: a máscara aparece menor e em contextos diferentes. Importante porque muitas fotos são de produto, com a máscara ocupando a imagem toda |
| Scale | Ultralytics | `scale=0.5` | Simula o soldador mais perto ou mais longe da câmera (2 a 6 m) |
| Translate / rotação leve | Ultralytics | `translate=0.1`, `degrees=5` | Câmeras instaladas com ângulos levemente diferentes; cabeça inclinada |
| Flip horizontal | Ultralytics | `fliplr=0.5` | Câmera à esquerda ou à direita do soldador |
| Flip vertical | – | `flipud=0` (desligado) | Ninguém solda de cabeça para baixo |
| Copy-paste | – | `copy_paste=0` | **Não tem efeito aqui:** no Ultralytics ele só funciona com máscaras de segmentação, e os dados têm só caixas |

As augmentations só valem para o treino. Validação e teste ficam com as imagens originais, então as métricas não são infladas.
`close_mosaic=10` desliga o mosaic nas 10 últimas épocas para o modelo se ajustar a imagens "normais".

### Modelos e tamanhos
- **YOLOv8n** (*nano*, ~3 M parâmetros): rápido em CPU, é o candidato para rodar no notebook do projeto.
- **YOLOv8s** (*small*, ~11 M parâmetros): mais preciso, cerca de 2 a 3 vezes mais lento.
- **YOLO26n** (*nano*, geração mais recente da Ultralytics): segundo a Ultralytics, mais rápido em CPU e um pouco
  mais preciso que as gerações anteriores. Entra como **comparação**: o projeto exige YOLOv8, e a escolha final
  considera o mAP (aqui) e o FPS medido no computador do projeto (Etapa 1.3).
- **imgsz 640 x 960**: 960 enxerga melhor objetos pequenos (soldador longe), mas custa ~2,2× mais processamento.

In [ ]:
from ultralytics import YOLO

PARAMETROS_AUG = dict(
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
    degrees=5.0, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, close_mosaic=10, mixup=0.0, copy_paste=0.0,
)


def nome_experimento(modelo: str, imgsz: int) -> str:
    return f"{Path(modelo).stem}_{imgsz}"


def ajustar_checkpoint(ultimo: Path, nome: str) -> None:
    # O checkpoint guarda os caminhos da máquina onde o treino começou (ex.: a pasta do Google Drive no Colab).
    # Antes de retomar, apontamos para as pastas desta sessão do Kaggle.
    ckpt = torch.load(ultimo, map_location="cpu", weights_only=False)
    args = dict(ckpt.get("train_args") or {})
    args.update({"project": str(PASTA_TREINOS), "name": nome, "data": str(DATA_YAML), "exist_ok": True})
    if "save_dir" in args:
        args["save_dir"] = str(PASTA_TREINOS / nome)
    ckpt["train_args"] = args
    torch.save(ckpt, ultimo)


def treinar(modelo: str, imgsz: int) -> Path:
    # Treina um experimento (ou retoma, se foi interrompido). Retorna a pasta do experimento.
    nome = nome_experimento(modelo, imgsz)
    pasta = PASTA_TREINOS / nome
    ultimo = pasta / "weights" / "last.pt"
    marcador = pasta / "CONCLUIDO.txt"     # gravado só quando o treino termina de verdade

    if marcador.exists():
        print(f"[{nome}] já concluído, pulando.")
        return pasta
    if ultimo.exists():
        print(f"[{nome}] retomando do último checkpoint...")
        ajustar_checkpoint(ultimo, nome)
        try:
            YOLO(str(ultimo)).train(resume=True)
        except Exception as erro:  # o Ultralytics recusa retomar um treino que já terminou
            print(f"[{nome}] nada a retomar ({erro}).")
        marcador.write_text("ok", encoding="utf-8")
        return pasta

    print(f"[{nome}] treinando...")
    YOLO(modelo).train(
        data=str(DATA_YAML), imgsz=imgsz, epochs=EPOCAS, patience=PACIENCIA,
        batch=-1,                 # tamanho de lote automático (~60% da memória da GPU)
        seed=SEMENTE, deterministic=True,
        project=str(PASTA_TREINOS), name=nome, exist_ok=True,
        workers=2, plots=True, **PARAMETROS_AUG,
    )
    marcador.write_text("ok", encoding="utf-8")
    return pasta


for modelo, imgsz in EXPERIMENTOS:
    treinar(modelo, imgsz)

## 9. Avaliação no conjunto de TESTE (RF32)

O conjunto de **teste** nunca foi usado no treino nem na escolha da melhor época, por isso é a medida honesta.

- **Precisão (P):** das máscaras que o modelo apontou, quantas eram mesmo máscaras. Precisão baixa significa alarmes falsos.
- **Recall (R):** das máscaras que existiam, quantas o modelo encontrou. Recall baixo significa que o sistema diria
  "sem máscara" para quem está de máscara.
- **mAP@0.5:** qualidade média considerando acerto quando a caixa sobrepõe ≥ 50% a caixa verdadeira. É a métrica das METAS DE VALIDAÇÃO (mínimo 0,60, meta boa 0,75).
- **mAP@0.5:0.95:** a mesma ideia, mas exigindo caixas cada vez mais precisas (mais rigorosa).

Também mostramos as métricas **por fonte** (Welding-PPE: cenas mais reais; Welding Helmet: mais fotos de produto),
para saber se o modelo vai bem nas duas.

**Taxa de falso positivo (rodada v3):** das imagens de teste **sem máscara**, em quantas o modelo "vê" uma máscara com
a confiança usada pelo sistema (0,45). É a medida direta do problema da webcam (rosto inclinado detectado como
máscara). Quanto menor, melhor; o mAP não mostra isso bem, porque ele é calculado sobre as máscaras que existem.

In [ ]:
METAS = {  # mAP@0.5 mínimo e meta boa (prompt do projeto)
    "mascara": (0.60, 0.75), "protecao_corpo": (0.60, 0.75),
    "luvas": (0.50, 0.65), "calcado": (0.40, 0.55),
}
split_avaliacao = "test"


def contar_instancias(pasta_labels: Path) -> dict[str, int]:
    cont = {c: 0 for c in CLASSES_MODELO}
    for arq in pasta_labels.glob("*.txt"):
        for linha in ler_label(arq):
            cont[CLASSES_MODELO[linha[0]]] += 1
    return cont


def subconjunto_por_fonte(prefixo: str) -> Path:
    # Cria um data.yaml cujo "test" contém só as imagens de teste de uma fonte (f0 ou f1).
    pasta = Path(f"{PASTA_TMP}/teste_{prefixo}")
    shutil.rmtree(pasta, ignore_errors=True)
    for sub in ("images", "labels"):
        (pasta / "test" / sub).mkdir(parents=True)
        for arq in (PASTA_DATASET / "test" / sub).glob(f"{prefixo}_*"):
            shutil.copy(arq, pasta / "test" / sub / arq.name)
    dados = {"path": str(pasta), "names": {i: n for i, n in enumerate(CLASSES_MODELO)},
             "train": str(PASTA_DATASET / "train" / "images"), "val": "test/images", "test": "test/images"}
    (pasta / "data.yaml").write_text(yaml.safe_dump(dados, allow_unicode=True, sort_keys=False), encoding="utf-8")
    return pasta


def avaliar(pasta_exp: Path, imgsz: int, data: Path, nome_aval: str) -> pd.DataFrame:
    # Avalia o best.pt e devolve uma tabela por classe.
    modelo = YOLO(str(pasta_exp / "weights" / "best.pt"))
    m = modelo.val(data=str(data / "data.yaml"), split="test", imgsz=imgsz, plots=True,
                   project=str(pasta_exp), name=nome_aval, exist_ok=True)
    instancias = contar_instancias(data / "test" / "labels")
    linhas = []
    for i, id_classe in enumerate(m.box.ap_class_index):
        p, r, map50, map5095 = m.box.class_result(i)
        classe = m.names[int(id_classe)]
        linhas.append({"classe": classe, "instancias": instancias.get(classe, 0),
                       "precisao": p, "recall": r, "mAP50": map50, "mAP50_95": map5095})
    df = pd.DataFrame(linhas).set_index("classe")
    if len(df) > 1:
        df.loc["TODAS (média)"] = [df["instancias"].sum(), m.box.mp, m.box.mr, m.box.map50, m.box.map]
    df.attrs["pasta_avaliacao"] = Path(m.save_dir)
    return df.round(3)


def situacao_meta(classe: str, map50: float) -> str:
    if classe not in METAS:
        return ""
    minimo, boa = METAS[classe]
    return "META BOA" if map50 >= boa else ("OK (mínimo)" if map50 >= minimo else "ABAIXO DO MÍNIMO")


subconjuntos = {"todas": PASTA_DATASET}
for prefixo in NOMES_FONTES:      # só fontes com máscaras no teste (as de negativos não têm mAP)
    if any(ler_label(a) for a in (PASTA_DATASET / "test" / "labels").glob(f"{prefixo}_*.txt")):
        subconjuntos[prefixo] = subconjunto_por_fonte(prefixo)

resultados, por_fonte = {}, []
for modelo, imgsz in EXPERIMENTOS:
    nome = nome_experimento(modelo, imgsz)
    for rotulo, data in subconjuntos.items():
        tabela = avaliar(PASTA_TREINOS / nome, imgsz, data, f"avaliacao_test_{rotulo}")
        tabela["situacao"] = [situacao_meta(c, v) for c, v in zip(tabela.index, tabela["mAP50"])]
        if rotulo == "todas":
            resultados[nome] = tabela
            tabela.to_csv(PASTA_TREINOS / nome / "metricas_por_classe_test.csv")
            print(f"\n===== {nome} (teste completo) =====")
            display(tabela)
        else:
            por_fonte.append(tabela.assign(experimento=nome, fonte=rotulo).reset_index())

if por_fonte:
    tabela_fontes = pd.concat(por_fonte).set_index(["experimento", "fonte", "classe"])
    print("\n===== Por fonte:", ", ".join(f"{p} = {n}" for p, n in NOMES_FONTES.items()), "=====")
    display(tabela_fontes)
    tabela_fontes.to_csv(PASTA_DRIVE / "metricas_por_fonte_test.csv")

CONF_SISTEMA = 0.45   # limiar da máscara no config.yaml do projeto


def taxa_falsos_positivos(pasta_exp: Path, imgsz: int) -> pd.DataFrame:
    # Fração das imagens de TESTE sem máscara em que o modelo detecta uma máscara (conf >= CONF_SISTEMA), por fonte.
    modelo = YOLO(str(pasta_exp / "weights" / "best.pt"))
    por_fonte = defaultdict(list)
    for arq in sorted((PASTA_DATASET / "test" / "labels").glob("*.txt")):
        if not ler_label(arq):
            por_fonte[arq.stem.split("_")[0]].append(next((PASTA_DATASET / "test" / "images").glob(arq.stem + ".*")))
    linhas = []
    for prefixo, imagens in sorted(por_fonte.items()):
        com_fp = 0
        for i in range(0, len(imagens), 32):
            resultados_lote = modelo.predict([str(p) for p in imagens[i:i + 32]], imgsz=imgsz, conf=CONF_SISTEMA,
                                             verbose=False)
            com_fp += sum(len(r.boxes) > 0 for r in resultados_lote)
        linhas.append({"fonte": prefixo, "nome": NOMES_FONTES.get(prefixo, prefixo),
                       "imagens_sem_mascara": len(imagens), "com_falso_positivo": com_fp,
                       "taxa_falso_positivo": round(com_fp / len(imagens), 3)})
    return pd.DataFrame(linhas)


tabelas_fp = [taxa_falsos_positivos(PASTA_TREINOS / nome_experimento(m, s), s).assign(experimento=nome_experimento(m, s))
              for m, s in EXPERIMENTOS]
tabela_fp = pd.concat(tabelas_fp).set_index(["experimento", "fonte"])
print("\n===== Falsos positivos nas imagens de teste SEM máscara (conf >= 0,45) =====")
display(tabela_fp)
tabela_fp.to_csv(PASTA_DRIVE / "falsos_positivos_test.csv")

### Comparação entre os experimentos (mAP@0.5)

In [ ]:
comparacao = pd.DataFrame({nome: t["mAP50"] for nome, t in resultados.items()})
display(comparacao)
comparacao.to_csv(PASTA_DRIVE / "comparacao_experimentos_mAP50.csv")
comparacao.T.plot.bar(rot=0, figsize=(8, 4), legend=len(comparacao) > 1, title="mAP@0.5 no teste por experimento")
plt.axhline(METAS["mascara"][0], ls="--", c="red", lw=0.8, label="mínimo")
plt.axhline(METAS["mascara"][1], ls="--", c="green", lw=0.8, label="meta boa")
plt.ylim(0, 1); plt.legend(); plt.tight_layout()
plt.savefig(PASTA_DRIVE / "comparacao_experimentos_mAP50.png"); plt.show()

### Matrizes de confusão

Com uma classe só, a matriz tem 2 linhas e 2 colunas: `mascara` e `background` (fundo).
- **mascara → background:** máscaras **não detectadas** (falsos negativos). O sistema diria "sem máscara" por engano.
- **background → mascara:** detecções de máscara onde **não havia** (falsos positivos). O sistema diria "com máscara" por engano.

In [ ]:
from IPython.display import Image as ImagemIPy

for nome, tabela in resultados.items():
    figura = tabela.attrs["pasta_avaliacao"] / "confusion_matrix_normalized.png"
    print(nome)
    display(ImagemIPy(filename=str(figura), width=550))

## 10. Escolha e exportação do modelo

Critério sugerido: o **menor** modelo/`imgsz` que atinge o **mínimo** da meta (mAP@0.5 ≥ 0,60), porque ele vai
rodar em **CPU**. Entre os de mesmo custo, o de **menor taxa de falso positivo** e, depois, o de maior mAP. O FPS real em CPU é medido depois, no seu computador
(`run.py --benchmark`, Etapa 1.3).

Ajuste `ESCOLHIDO` se preferir outro experimento.

In [ ]:
custo = {nome_experimento(m, s): (("yolov8s" in m) * 2 + (s > 640)) for m, s in EXPERIMENTOS}
map_mascara = {n: float(t.loc["mascara", "mAP50"]) for n, t in resultados.items()}
atingem = [n for n, v in map_mascara.items() if v >= METAS["mascara"][0]]

for n in sorted(map_mascara, key=map_mascara.get, reverse=True):
    print(f"{n:15s}  mAP50 mascara = {map_mascara[n]:.3f}  {'(atinge o mínimo)' if n in atingem else ''}")

fp_medio = {n: float(tabela_fp.loc[n]["taxa_falso_positivo"].mean()) for n in resultados}
for n in resultados:
    print(f"{n:15s}  taxa média de falso positivo = {fp_medio[n]:.3f}")

if atingem:   # mais barato entre os que atingem; empate -> menos falsos positivos -> maior mAP
    ESCOLHIDO = min(atingem, key=lambda n: (custo[n], round(fp_medio[n], 2), -map_mascara[n]))
else:
    ESCOLHIDO = max(map_mascara, key=map_mascara.get)
    print("\nNenhum experimento atingiu o mínimo; escolhendo o de maior mAP.")
print("\nModelo escolhido:", ESCOLHIDO)

destino = PASTA_DRIVE / "epi_solda_best.pt"
shutil.copy(PASTA_TREINOS / ESCOLHIDO / "weights" / "best.pt", destino)

# Pacote para levar ao projeto: modelo + métricas + gráficos principais
pacote = Path(f"{PASTA_TMP}/resultado_treino_{RODADA}"); shutil.rmtree(pacote, ignore_errors=True); pacote.mkdir()
shutil.copy(destino, pacote)
for arquivo in ["balanceamento_classes.csv", "balanceamento_classes.png", "metricas_por_fonte_test.csv",
                "falsos_positivos_test.csv", "comparacao_experimentos_mAP50.csv", "comparacao_experimentos_mAP50.png"]:
    if (PASTA_DRIVE / arquivo).exists():
        shutil.copy(PASTA_DRIVE / arquivo, pacote)
for nome, tabela in resultados.items():
    (pacote / nome).mkdir()
    for origem in [PASTA_TREINOS / nome / "metricas_por_classe_test.csv", PASTA_TREINOS / nome / "results.csv",
                   tabela.attrs["pasta_avaliacao"] / "confusion_matrix_normalized.png"]:
        if origem.exists():            # um arquivo faltando não impede a exportação do modelo
            shutil.copy(origem, pacote / nome)
        else:
            print(f"Aviso: {origem} não encontrado; seguindo sem ele.")
fontes_txt = "\n".join(f"  f{i}: {f['nome']} ({f['workspace']}/{f['projeto']} v{f['versao']})" for i, f in enumerate(FONTES))
(pacote / "LEIA-ME.txt").write_text(
    f"Rodada: {RODADA}\nModelo escolhido: {ESCOLHIDO}\nClasses: {CLASSES_MODELO}\n"
    f"Imagens originais por split: { {s: len(v) for s, v in divisao.items()} }\n"
    f"Fontes (Roboflow Universe, CC BY 4.0):\n{fontes_txt}\n",
    encoding="utf-8")
zip_final = shutil.make_archive(f"/kaggle/working/resultado_treino_{RODADA}", "zip", pacote)
print("Pronto! Baixe", Path(zip_final).name, "na aba Output desta versão do notebook.")

## 11. Próximos passos (no seu computador)

1. Na página da versão do notebook, abra a aba **Output** e baixe `resultado_treino_v3.zip`. Descompacte.
2. Copie `epi_solda_best.pt` para a pasta `models/` do projeto (`Z:\Documentos\SafeWork\models\`),
   **substituindo** o da rodada anterior. Os arquivos `.pt` ficam fora do git (estão no `.gitignore`).
3. Coloque o restante do zip em `docs/resultados_treino/v3/` e me avise: eu analiso as métricas contra as
   METAS DE VALIDAÇÃO.